In [1]:
import pandas as pd
import os
from pathlib import Path
import re
from datetime import datetime
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import rasterio
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from torchmetrics.classification import MulticlassF1Score
from glob import glob
import torch
import random
import torch.nn.functional as F
import numpy as np
import torch.nn as nn

C:\Users\lolke\miniconda3\envs\cosmohack\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
s1_pre_folder = './bs/sentinel1_pre/'
s1_post_folder = './bs/sentinel1_post/'
s2_pre_folder = './bs/sentinel2_pre/'
s2_post_folder = './bs/sentinel2_post/'
aux_folder = './bs/aux/'

ds_metadata = pd.read_csv("./bs/meta.csv")

In [3]:
import rasterio
import numpy as np

idx = 21
s1_pre_prefix = "_Sentinel-1_pre"
s1_post_prefix = "_Sentinel-1_post"
s2_pre_prefix = "_Sentinel-2_pre"
s2_post_prefix = "_Sentinel-2_post"
aux_prefix = "_AUX"
path = aux_folder + ds_metadata.iloc[idx]["chip_id"] + aux_prefix + ".tif"

with rasterio.open(path) as src:
    s1_pre = src.read().astype(np.float32)  # shape: (C, H, W)
    print("shape:", s1_pre.shape, "dtype:", s1_pre.dtype)
    print("min/max:", s1_pre.min(), s1_pre.max())

shape: (3, 512, 512) dtype: float32
min/max: 4.0 141.0


In [4]:
import numpy as np
import rasterio

path = "bs/aux/BS_tr_000003_AUX.tif"  # подставь реальный путь

with rasterio.open(path) as src:
    aux = src.read().astype(np.float32)
    print("shape:", aux.shape, "dtype в файле:", src.dtypes, "nodata:", src.nodata)

for c in range(aux.shape[0]):
    ch = aux[c]
    finite = ch[np.isfinite(ch)]
    n_unique = len(np.unique(finite))
    print(f"\n--- канал {c} ---")
    print(f"min={finite.min():.2f} max={finite.max():.2f} mean={finite.mean():.2f}")
    print(f"n_unique={n_unique}")
    if n_unique <= 20:
        vals, counts = np.unique(finite, return_counts=True)
        print("уникальные значения и их частоты:", dict(zip(vals.tolist(), counts.tolist())))
    else:
        print("перцентили [1,25,50,75,99]:", np.percentile(finite, [1, 25, 50, 75, 99]))

shape: (3, 512, 512) dtype в файле: ('int16', 'int16', 'int16') nodata: None

--- канал 0 ---
min=-13.00 max=4.00 mean=-5.17
n_unique=18
уникальные значения и их частоты: {-13.0: 27532, -12.0: 523, -11.0: 2766, -10.0: 6764, -9.0: 14710, -8.0: 11386, -7.0: 13435, -6.0: 21161, -5.0: 25662, -4.0: 28859, -3.0: 36360, -2.0: 39721, -1.0: 21101, 0.0: 9513, 1.0: 1501, 2.0: 836, 3.0: 312, 4.0: 2}

--- канал 1 ---
min=1.00 max=1.00 mean=1.00
n_unique=1
уникальные значения и их частоты: {1.0: 262144}

--- канал 2 ---
min=20.00 max=90.00 mean=34.33
n_unique=6
уникальные значения и их частоты: {20.0: 104, 30.0: 227218, 40.0: 58, 60.0: 31517, 80.0: 469, 90.0: 2778}


In [5]:
import random
import numpy as np
import rasterio
import torch
from torch.utils.data import Dataset

S2_SCALE = 10000.0
S1_SCALE = 100.0

S1_CLIP_RANGES = {"VV": (-25.0, 0.0), "VH": (-32.5, 0.0)}

S2_BAND_IDX = {
    "B2": 0, "B3": 1, "B4": 2, "B5": 3, "B6": 4,
    "B7": 5, "B8A": 6, "B11": 7, "B12": 8, "SCL": 9,
}

S2_DN_CLIP = (0.0, 10000.0)

SCL_INVALID = {0, 1, 3, 6, 8, 9, 10}

WORLDCOVER_CODES = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
CODE_TO_CLASS_IDX = {code: i for i, code in enumerate(WORLDCOVER_CODES)}
N_LANDCOVER_CLASSES = len(WORLDCOVER_CODES)

FOLDERS = {
    "s1_pre": ("./bs/sentinel1_pre/", "_Sentinel-1_pre"),
    "s1_post": ("./bs/sentinel1_post/", "_Sentinel-1_post"),
    "s2_pre": ("./bs/sentinel2_pre/", "_Sentinel-2_pre"),
    "s2_post": ("./bs/sentinel2_post/", "_Sentinel-2_post"),
    "aux": ("./bs/aux/", "_AUX"),
    "mask": ("./bs/masks/", "_MASK"),  # уточни реальную папку/суффикс маски у себя
}


def build_path(chip_id, kind):
    folder, suffix = FOLDERS[kind]
    return folder + chip_id + suffix + ".tif"


def load_tif(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.float32)

def load_s2(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
    arr = np.clip(arr, S2_DN_CLIP[0], S2_DN_CLIP[1])
    return arr

def normalize_s1_db_scaled(arr, band_indices={"VV": 0, "VH": 1}, scale=S1_SCALE):
    h, w = arr.shape[1], arr.shape[2]
    out = np.zeros((2, h, w), dtype=np.float32)
    for out_i, band in enumerate(("VV", "VH")):
        src_i = band_indices[band]
        band_db = arr[src_i] / scale
        lo, hi = S1_CLIP_RANGES[band]
        band_db = np.clip(band_db, lo, hi)
        out[out_i] = (band_db - lo) / (hi - lo)
    return out


def process_s1_pair(s1_pre_raw, s1_post_raw):
    return normalize_s1_db_scaled(s1_pre_raw), normalize_s1_db_scaled(s1_post_raw)


def build_valid_mask_scl(scl_pre, scl_post):
    invalid_pre = np.isin(scl_pre, list(SCL_INVALID))
    invalid_post = np.isin(scl_post, list(SCL_INVALID))
    return (~(invalid_pre | invalid_post)).astype(np.float32)


def process_aux_3ch(aux_arr, elevation_range):
    """aux_arr: (3, H, W) -> [dem, slope, land_cover_class]
    Порядок каналов подтверждён через rasterio descriptions: (dem, slope, landcover).
    Никакой маски валидности в AUX нет — валидность определяется только через SCL.
    """
    dem, slope, land_cover = aux_arr
    dem_norm = np.clip(
        (dem - elevation_range[0]) / (elevation_range[1] - elevation_range[0]),
        0, 1
    ).astype(np.float32)
    slope_norm = np.clip(slope / 90.0, 0, 1).astype(np.float32)  # градусы 0-90 -> [0,1]
    return dem_norm[None, ...], slope_norm[None, ...], land_cover.astype(np.int64)


def landcover_to_onehot(land_cover_raw):
    h, w = land_cover_raw.shape
    onehot = np.zeros((N_LANDCOVER_CLASSES + 1, h, w), dtype=np.float32)

    known = np.zeros((h, w), dtype=bool)

    for code, class_idx in CODE_TO_CLASS_IDX.items():
        class_mask = land_cover_raw == code
        onehot[class_idx] = class_mask.astype(np.float32)
        known |= class_mask

    onehot[-1] = (~known).astype(np.float32)
    return onehot

IGNORE_INDEX = 255

class CosmohackDataset(Dataset):
    def __init__(self, df, is_train=False):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def remap_mask(self, mask):
        return mask.astype(np.int64)  # BS-маска уже 0-3 согласно постановке кейса

    def apply_augmentations(self, x, mask):
        if random.random() > 0.5:
            x = np.flip(x, axis=2)
            mask = np.flip(mask, axis=1)
        if random.random() > 0.5:
            x = np.flip(x, axis=1)
            mask = np.flip(mask, axis=0)
        k = random.randint(0, 3)
        if k > 0:
            x = np.rot90(x, k, axes=(1, 2))
            mask = np.rot90(mask, k, axes=(0, 1))
        return np.ascontiguousarray(x), np.ascontiguousarray(mask)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        chip_id = row["chip_id"]
    
        s2_pre = load_s2(build_path(chip_id, "s2_pre")) / S2_SCALE
        s2_post = load_s2(build_path(chip_id, "s2_post")) / S2_SCALE
    
        idx_red = S2_BAND_IDX["B4"]
        idx_nir = S2_BAND_IDX["B8A"]
        idx_swir1 = S2_BAND_IDX["B11"]
        idx_swir2 = S2_BAND_IDX["B12"]
        idx_scl = S2_BAND_IDX["SCL"]
    
        pre_red = s2_pre[idx_red][None, ...]
        pre_nir = s2_pre[idx_nir][None, ...]
        pre_swir1 = s2_pre[idx_swir1][None, ...]
        pre_swir2 = s2_pre[idx_swir2][None, ...]
    
        post_red = s2_post[idx_red][None, ...]
        post_nir = s2_post[idx_nir][None, ...]
        post_swir1 = s2_post[idx_swir1][None, ...]
        post_swir2 = s2_post[idx_swir2][None, ...]
    
        scl_pre = (s2_pre[idx_scl] * S2_SCALE).round()
        scl_post = (s2_post[idx_scl] * S2_SCALE).round()
    
        pre_nbr1 = np.clip(
            (pre_nir - pre_swir1) / (pre_nir + pre_swir1 + 1e-8),
            -1.0,
            1.0,
        )
    
        post_nbr1 = np.clip(
            (post_nir - post_swir1) / (post_nir + post_swir1 + 1e-8),
            -1.0,
            1.0,
        )
    
        pre_nbr2 = np.clip(
            (pre_nir - pre_swir2) / (pre_nir + pre_swir2 + 1e-8),
            -1.0,
            1.0,
        )
    
        post_nbr2 = np.clip(
            (post_nir - post_swir2) / (post_nir + post_swir2 + 1e-8),
            -1.0,
            1.0,
        )
    
        pre_ndvi = np.clip(
            (pre_nir - pre_red) / (pre_nir + pre_red + 1e-8),
            -1.0,
            1.0,
        )
    
        post_ndvi = np.clip(
            (post_nir - post_red) / (post_nir + post_red + 1e-8),
            -1.0,
            1.0,
        )
    
        dndvi = np.clip(pre_ndvi - post_ndvi, -2.0, 2.0)
    
        dnbr1 = np.clip(pre_nbr1 - post_nbr1, -2.0, 2.0)
        dnbr2 = np.clip(pre_nbr2 - post_nbr2, -2.0, 2.0)
    
        rdnbr = dnbr1 / (np.sqrt(np.abs(pre_nbr1)) + 1e-8)
        rdnbr = np.clip(rdnbr, -6.0, 6.0) / 6.0
    
        scl_valid_mask = build_valid_mask_scl(scl_pre, scl_post)[None, ...]
    
        s1_pre_raw = load_tif(build_path(chip_id, "s1_pre"))
        s1_post_raw = load_tif(build_path(chip_id, "s1_post"))
    
        pre_norm_s1, post_norm_s1 = process_s1_pair(
            s1_pre_raw,
            s1_post_raw,
        )
    
        aux_raw = load_tif(build_path(chip_id, "aux"))
    
        dem_norm, slope_norm, land_cover_raw = process_aux_3ch(
            aux_raw,
            elevation_range=(-26, 366),
        )
    
        land_cover_onehot = landcover_to_onehot(land_cover_raw)
    
        combined_valid_mask = scl_valid_mask
        x = np.concatenate([
            pre_red,
            pre_nir,
            pre_swir1,
    
            post_red,
            post_nir,
            post_swir1,
    
            dndvi,
            dnbr1,
            dnbr2,
            rdnbr,
    
            pre_norm_s1,
            post_norm_s1,
    
            dem_norm,
            slope_norm,
    
            land_cover_onehot,
    
            combined_valid_mask,
        ], axis=0).astype(np.float32)
        mask_path = build_path(chip_id, "mask")
        mask = load_tif(mask_path).astype(np.int64)[0]
        mask = self.remap_mask(mask)
    
        valid_pixels = combined_valid_mask[0] > 0.5
    
        mask[~valid_pixels] = IGNORE_INDEX
    
        optical_channels = slice(0, 10)
    
        x[optical_channels, ~valid_pixels] = 0.0
    
        x = np.nan_to_num(
            x,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        ).astype(np.float32)
    
        if self.is_train:
            x, mask = self.apply_augmentations(x, mask)
    
        x_tensor = torch.from_numpy(x).float()
        mask_tensor = torch.from_numpy(mask).long()
    
        return x_tensor, mask_tensor

In [6]:
device = "cuda"

f1_metric = MulticlassF1Score(
    num_classes=4,
    average=None,
    ignore_index=255
).to(device)

In [7]:
class ConvBlock(torch.nn.Module):
    def __init__(self, ch_in, ch_out):
        super().__init__()
        self.conv = nn.Sequential(
                                  nn.Conv2d(ch_in, ch_out,
                                            kernel_size=3, stride=1,
                                            padding=1, bias=True),
                                  nn.BatchNorm2d(ch_out),
                                  nn.ReLU(inplace=True),
                                  nn.Conv2d(ch_out, ch_out,
                                            kernel_size=3, stride=1,
                                            padding=1, bias=True),
                                  nn.BatchNorm2d(ch_out),
                                  nn.ReLU(inplace=True),
        )
        
    def forward(self, x):
        x = self.conv(x)
        return x

In [8]:
class UpConvBlock(torch.nn.Module):
    def __init__(self, ch_in, ch_out):
        super().__init__()
        self.up = nn.Sequential(
                                nn.Upsample(scale_factor=2),
                                nn.Conv2d(ch_in, ch_out,
                                         kernel_size=3,stride=1,
                                         padding=1, bias=True),
                                nn.BatchNorm2d(ch_out),
                                nn.ReLU(inplace=True),
        )
        
    def forward(self, x):
        x = x = self.up(x)
        return x

In [9]:
class AttentionBlock(torch.nn.Module):
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        
        self.w_g = nn.Sequential(
                                nn.Conv2d(f_g, f_int,
                                         kernel_size=1, stride=1,
                                         padding=0, bias=True),
                                nn.BatchNorm2d(f_int)
        )
        
        self.w_x = nn.Sequential(
                                nn.Conv2d(f_l, f_int,
                                         kernel_size=1, stride=1,
                                         padding=0, bias=True),
                                nn.BatchNorm2d(f_int)
        )
        
        self.psi = nn.Sequential(
                                nn.Conv2d(f_int, 1,
                                         kernel_size=1, stride=1,
                                         padding=0,  bias=True),
                                nn.BatchNorm2d(1),
                                nn.Sigmoid(),
        )
        
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, g, x):
        g1 = self.w_g(g)
        x1 = self.w_x(x)
        psi = self.relu(g1+x1)
        psi = self.psi(psi)
        
        return psi*x

In [10]:
class AttentionUNet(torch.nn.Module):
    def __init__(self, n_classes=4, in_channel=29, out_channel=4):
        super().__init__() 
        
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv1 = ConvBlock(ch_in=in_channel, ch_out=64)
        self.conv2 = ConvBlock(ch_in=64, ch_out=128)
        self.conv3 = ConvBlock(ch_in=128, ch_out=256)
        self.conv4 = ConvBlock(ch_in=256, ch_out=512)
        self.conv5 = ConvBlock(ch_in=512, ch_out=1024)
        
        self.up5 = UpConvBlock(ch_in=1024, ch_out=512)
        self.att5 = AttentionBlock(f_g=512, f_l=512, f_int=256)
        self.upconv5 = ConvBlock(ch_in=1024, ch_out=512)
        
        self.up4 = UpConvBlock(ch_in=512, ch_out=256)
        self.att4 = AttentionBlock(f_g=256, f_l=256, f_int=128)
        self.upconv4 = ConvBlock(ch_in=512, ch_out=256)
        
        self.up3 = UpConvBlock(ch_in=256, ch_out=128)
        self.att3 = AttentionBlock(f_g=128, f_l=128, f_int=64)
        self.upconv3 = ConvBlock(ch_in=256, ch_out=128)
        
        self.up2 = UpConvBlock(ch_in=128, ch_out=64)
        self.att2 = AttentionBlock(f_g=64, f_l=64, f_int=32)
        self.upconv2 = ConvBlock(ch_in=128, ch_out=64)
        
        self.conv_1x1 = nn.Conv2d(64, out_channel,
                                  kernel_size=1, stride=1, padding=0)
        
    def forward(self, x):
        # encoder
        x1 = self.conv1(x)
        
        x2 = self.maxpool(x1)
        x2 = self.conv2(x2)
        
        x3 = self.maxpool(x2)
        x3 = self.conv3(x3)
        
        x4 = self.maxpool(x3)
        x4 = self.conv4(x4)
        
        x5 = self.maxpool(x4)
        x5 = self.conv5(x5)
        
        # decoder + concat
        d5 = self.up5(x5)
        x4 = self.att5(g=d5, x=x4)
        d5 = torch.concat((x4, d5), dim=1)
        d5 = self.upconv5(d5)
        
        d4 = self.up4(d5)
        x3 = self.att4(g=d4, x=x3)
        d4 = torch.concat((x3, d4), dim=1)
        d4 = self.upconv4(d4)
        
        d3 = self.up3(d4)
        x2 = self.att3(g=d3, x=x2)
        d3 = torch.concat((x2, d3), dim=1)
        d3 = self.upconv3(d3)
        
        d2 = self.up2(d3)
        x1 = self.att2(g=d2, x=x1)
        d2 = torch.concat((x1, d2), dim=1)
        d2 = self.upconv2(d2)
        
        d1 = self.conv_1x1(d2)
        
        return d1

In [18]:
from typing import Optional, Sequence
from torch import Tensor

class DiceLoss(nn.Module):
    def __init__(self, num_classes, smooth=1e-6, ignore_index=255):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, target):
        valid_mask = (target != self.ignore_index)

        safe_target = target.clone()
        safe_target[~valid_mask] = 0  # временно подменяем, чтобы one_hot не упал

        probs = F.softmax(logits, dim=1)
        target_oh = F.one_hot(safe_target, self.num_classes).permute(0, 3, 1, 2).float()

        valid_mask = valid_mask.unsqueeze(1).float()  # [B, 1, H, W]
        probs = probs * valid_mask
        target_oh = target_oh * valid_mask

        dims = (0, 2, 3)
        intersection = torch.sum(probs * target_oh, dims)
        cardinality = torch.sum(probs + target_oh, dims)
        dice = (2. * intersection + self.smooth) / (cardinality + self.smooth)
        return 1 - dice.mean()

class CombinedLoss(nn.Module):
    def __init__(self, class_weights, num_classes, ce_w=0.5, dice_w=0.5, ignore_index=255):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights, ignore_index=ignore_index)
        self.dice = DiceLoss(num_classes, ignore_index=ignore_index)
        self.ce_w, self.dice_w = ce_w, dice_w

    def forward(self, logits, target):
        return self.ce_w * self.ce(logits, target) + self.dice_w * self.dice(logits, target)

class FocalLoss(nn.Module):
    """ Focal Loss, as described in https://arxiv.org/abs/1708.02002.

    It is essentially an enhancement to cross entropy loss and is
    useful for classification tasks when there is a large class imbalance.
    x is expected to contain raw, unnormalized scores for each class.
    y is expected to contain class labels.

    Shape:
        - x: (batch_size, C) or (batch_size, C, d1, d2, ..., dK), K > 0.
        - y: (batch_size,) or (batch_size, d1, d2, ..., dK), K > 0.
    """

    def __init__(self,
                 alpha: Optional[Tensor] = None,
                 gamma: float = 0.,
                 reduction: str = 'mean',
                 ignore_index: int = -100):
        """Constructor.

        Args:
            alpha (Tensor, optional): Weights for each class. Defaults to None.
            gamma (float, optional): A constant, as described in the paper.
                Defaults to 0.
            reduction (str, optional): 'mean', 'sum' or 'none'.
                Defaults to 'mean'.
            ignore_index (int, optional): class label to ignore.
                Defaults to -100.
        """
        if reduction not in ('mean', 'sum', 'none'):
            raise ValueError(
                'Reduction must be one of: "mean", "sum", "none".')

        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.reduction = reduction

        self.nll_loss = nn.NLLLoss(
            weight=alpha, reduction='none', ignore_index=ignore_index)

    def __repr__(self):
        arg_keys = ['alpha', 'gamma', 'ignore_index', 'reduction']
        arg_vals = [self.__dict__[k] for k in arg_keys]
        arg_strs = [f'{k}={v!r}' for k, v in zip(arg_keys, arg_vals)]
        arg_str = ', '.join(arg_strs)
        return f'{type(self).__name__}({arg_str})'

    def forward(self, x: Tensor, y: Tensor) -> Tensor:
        if x.ndim > 2:
            # (N, C, d1, d2, ..., dK) --> (N * d1 * ... * dK, C)
            c = x.shape[1]
            x = x.permute(0, *range(2, x.ndim), 1).reshape(-1, c)
            # (N, d1, d2, ..., dK) --> (N * d1 * ... * dK,)
            y = y.view(-1)

        unignored_mask = y != self.ignore_index
        y = y[unignored_mask]
        if y.numel() == 0:
            return x.sum() * 0.0
        x = x[unignored_mask]

        # compute weighted cross entropy term: -alpha * log(pt)
        # (alpha is already part of self.nll_loss)
        log_p = F.log_softmax(x, dim=-1)
        ce = self.nll_loss(log_p, y)

        # get true class column from each row
        all_rows = torch.arange(len(x), device=x.device)
        log_pt = log_p[all_rows, y]

        # compute focal term: (1 - pt)^gamma
        pt = log_pt.exp()
        focal_term = (1 - pt)**self.gamma

        # the full loss: -alpha * ((1 - pt)^gamma) * log(pt)
        loss = focal_term * ce

        if self.reduction == 'mean':
            loss = loss.mean()
        elif self.reduction == 'sum':
            loss = loss.sum()

        return loss


def focal_loss(alpha: Optional[Sequence] = None,
               gamma: float = 2,
               reduction: str = 'mean',
               ignore_index: int = 255,
               device='cuda',
               dtype=torch.float32) -> FocalLoss:
    """Factory function for FocalLoss.

    Args:
        alpha (Sequence, optional): Weights for each class. Will be converted
            to a Tensor if not None. Defaults to None.
        gamma (float, optional): A constant, as described in the paper.
            Defaults to 0.
        reduction (str, optional): 'mean', 'sum' or 'none'.
            Defaults to 'mean'.
        ignore_index (int, optional): class label to ignore.
            Defaults to -100.
        device (str, optional): Device to move alpha to. Defaults to 'cpu'.
        dtype (torch.dtype, optional): dtype to cast alpha to.
            Defaults to torch.float32.

    Returns:
        A FocalLoss object
    """
    if alpha is not None:
        if not isinstance(alpha, Tensor):
            alpha = torch.tensor(alpha)
        alpha = alpha.to(device=device, dtype=dtype)

    fl = FocalLoss(
        alpha=alpha,
        gamma=gamma,
        reduction=reduction,
        ignore_index=ignore_index)
    return fl

In [12]:
from torch.utils.data import WeightedRandomSampler

def calculate_sample_weights(df):   
    global_class_counts = np.zeros(4, dtype=np.int64)
    chip_stats = []
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        chip_id = row["chip_id"]
        mask_path = build_path(chip_id, "mask") 
        
        with rasterio.open(mask_path) as src:
            mask = src.read(1)
            
        counts = np.bincount(mask.ravel(), minlength=4)
        global_class_counts += counts
        chip_stats.append(counts)
    
    class_weights = 1.0 / (global_class_counts + 1)
    class_weights = class_weights / np.sum(class_weights)
    
    sample_weights = []
    for counts in chip_stats:
        weight = np.sum(counts * class_weights)
        sample_weights.append(weight)
        
    return sample_weights

def calculate_class_weights(df):
    counts = np.zeros(4, dtype=np.int64)

    for _, row in tqdm(df.iterrows(), total=len(df)):
        chip_id = row["chip_id"]

        mask = load_tif(build_path(chip_id, "mask"))[0].astype(np.int64)

        s2_pre = load_s2(build_path(chip_id, "s2_pre")) / S2_SCALE
        s2_post = load_s2(build_path(chip_id, "s2_post")) / S2_SCALE

        scl_pre = (s2_pre[S2_BAND_IDX["SCL"]] * S2_SCALE).round()
        scl_post = (s2_post[S2_BAND_IDX["SCL"]] * S2_SCALE).round()

        valid = build_valid_mask_scl(scl_pre, scl_post).astype(bool)

        valid_mask_values = mask[valid]
        counts += np.bincount(valid_mask_values, minlength=4)

    weights = counts.sum() / (len(counts) * np.maximum(counts, 1))
    weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float32, device=device)

In [20]:
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

torch.backends.cudnn.benchmark = True
NUM_EPOCHS = 25

train_df, val_df = train_test_split(ds_metadata, test_size=0.1)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_df, test_df = train_test_split(train_df, test_size=0.12)
train_df = train_df.reset_index(drop=True)

train_sample_weights = calculate_sample_weights(train_df)

sampler = WeightedRandomSampler(
    weights=train_sample_weights,
    num_samples=len(train_sample_weights),
    replacement=True
)

class_weights = calculate_class_weights(train_df)

print(f"Train: {train_df.shape} \nVal: {val_df.shape} \nTest: {test_df.shape}")

train_ds = CosmohackDataset(train_df, is_train=True)
val_ds = CosmohackDataset(val_df)
test_ds = CosmohackDataset(test_df)

model = AttentionUNet().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=0.1, nesterov=True)

criterion = CombinedLoss(class_weights=class_weights, num_classes=4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda", enabled=(amp_dtype == torch.float16))

train_loader = DataLoader(
    train_ds, batch_size=4,
    num_workers=0, pin_memory=True, sampler=sampler)

val_loader = DataLoader(
    val_ds, batch_size=4, shuffle=False, 
    num_workers=0, pin_memory=True)

test_loader = DataLoader(
    test_ds, batch_size=4, shuffle=False, 
    num_workers=0, pin_memory=True)

best_miou = 0.0

for epoch in tqdm(range(NUM_EPOCHS)):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(device, memory_format=torch.channels_last, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            outputs = model(images)
            loss = criterion(outputs, masks)
        
        if amp_dtype == torch.bfloat16:
            loss.backward()
            optimizer.step()
        else:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        train_loss += loss.detach().item()
            
    model.eval()
    f1_metric.reset()
    val_loss = 0.0
    total_intersection = torch.zeros(4, device=device)
    total_union = torch.zeros(4, device=device)
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device, memory_format=torch.channels_last, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                outputs = model(images)
                val_loss += criterion(outputs, masks).item()
            preds = outputs.argmax(dim=1)
            f1_metric.update(outputs, masks.long())

            valid = (masks != 255)
            
            for c in range(4):
                p_c = (preds == c) & valid
                m_c = (masks == c) & valid
                total_intersection[c] += (p_c & m_c).sum()
                total_union[c] += (p_c | m_c).sum()

    mean_ious = torch.where(total_union > 0, total_intersection / total_union, torch.full_like(total_union, float("nan"))).cpu().numpy()
    miou_all = np.nanmean(mean_ious)
    miou_target = np.nanmean(mean_ious[1:])
    f1_epoch = f1_metric.compute().cpu().numpy()
    val_loss /= len(val_loader)
    scheduler.step()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss:.4f} | F1-score: {f1_epoch}\niou: {mean_ious.round(4)}, miou={miou_all}, miou target={miou_target}")
    if miou_target > best_miou:
        best_miou = miou_target
        torch.save(model.state_dict(), f"weighted_comboloss_sampler.pt")
        print(f"Чекпоинт сохранен: target mIoU = {best_miou:.4f}")

100%|██████████| 176/176 [00:12<00:00, 14.31it/s]


Train: (176, 26) 
Val: (23, 26) 
Test: (25, 26)


  4%|▍         | 1/25 [00:54<21:49, 54.57s/it]

Epoch 1/25 | Train Loss: 0.9200 | Val Loss: 1.0219 | F1-score: [0.9237997  0.1378289  0.41145158 0.32001343]
iou: [0.8584 0.074  0.259  0.1905], miou=0.3454754948616028, miou target=0.17450398206710815
Чекпоинт сохранен: target mIoU = 0.1745


  8%|▊         | 2/25 [01:48<20:48, 54.27s/it]

...

Epoch 23/25 | Train Loss: 0.4480 | Val Loss: 0.4234 | F1-score: [0.9365035  0.52223647 0.69428235 0.8570083 ]
iou: [0.8806 0.3534 0.5317 0.7498], miou=0.6288761496543884, miou target=0.544971764087677


 96%|█████████▌| 24/25 [22:10<00:54, 54.27s/it]

Epoch 24/25 | Train Loss: 0.4365 | Val Loss: 0.4108 | F1-score: [0.93878835 0.52825755 0.699877   0.86440694]
iou: [0.8846 0.3589 0.5383 0.7612], miou=0.6357704997062683, miou target=0.5528145432472229
Чекпоинт сохранен: target mIoU = 0.5528


100%|██████████| 25/25 [23:03<00:00, 55.36s/it]

Epoch 25/25 | Train Loss: 0.4545 | Val Loss: 0.4506 | F1-score: [0.9402033  0.5354239  0.6915379  0.84701496]
iou: [0.8872 0.3656 0.5285 0.7346], miou=0.6289693117141724, miou target=0.5429075360298157


In [5]:
import numpy as np
import rasterio
import glob

def load_mask(path):
    with rasterio.open(path) as src:
        return src.read().astype(np.int64)[0]

def class_distribution(mask_paths, n_classes=4):
    counts = np.zeros(n_classes, dtype=np.int64)
    for p in mask_paths:
        mask = load_mask(p)
        for c in range(n_classes):
            counts[c] += (mask == c).sum()
    return counts

mask_paths = glob.glob("./bs/masks/*.tif")
counts = class_distribution(mask_paths)
shares = counts / counts.sum()

for c in range(len(counts)):
    print(f"class {c}: {counts[c]} px ({shares[c]:.4%})")

class 0: 52811255 px (89.9370%)
class 1: 2362871 px (4.0239%)
class 2: 2192271 px (3.7334%)
class 3: 1353859 px (2.3056%)


In [22]:
def evaluate_model(model, loader, device, num_classes=4, ignore_index=255):
    model.eval()

    total_intersection = torch.zeros(num_classes, device=device)
    total_union = torch.zeros(num_classes, device=device)

    f1_metric = MulticlassF1Score(
        num_classes=num_classes,
        average=None,
        ignore_index=ignore_index,
    ).to(device)

    f1_metric.reset()

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(
                device,
                memory_format=torch.channels_last,
                non_blocking=True,
            )

            masks = masks.to(
                device,
                non_blocking=True,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=amp_dtype,
            ):
                outputs = model(images)

            preds = outputs.argmax(dim=1)

            f1_metric.update(outputs, masks)

            valid = masks != ignore_index

            for c in range(num_classes):
                pred_c = (preds == c) & valid
                target_c = (masks == c) & valid

                total_intersection[c] += (pred_c & target_c).sum()
                total_union[c] += (pred_c | target_c).sum()

    iou_per_class = torch.where(
        total_union > 0,
        total_intersection / total_union,
        torch.full_like(total_union, float("nan")),
    ).cpu().numpy()

    f1_per_class = f1_metric.compute().cpu().numpy()

    return {
        "iou_per_class": iou_per_class,
        "f1_per_class": f1_per_class,
        "miou_all": np.nanmean(iou_per_class),
        "miou_target": np.nanmean(iou_per_class[1:]),
    }
trained_model = AttentionUNet().to(device)
state_dict = torch.load("weighted_comboloss_sampler.pt")
trained_model.load_state_dict(state_dict)
trained_model.eval()
test_metrics = evaluate_model(
    model=trained_model,
    loader=test_loader,
    device=device,
    num_classes=4,
    ignore_index=IGNORE_INDEX,
)

print("Test F1 per class:", np.round(test_metrics["f1_per_class"], 4))
print("Test IoU per class:", np.round(test_metrics["iou_per_class"], 4))
print("Test mIoU all:", round(test_metrics["miou_all"], 4))
print("Test target mIoU:", round(test_metrics["miou_target"], 4))

Test F1 per class: [0.9368 0.3637 0.7188 0.7752]
Test IoU per class: [0.8811 0.2222 0.5611 0.633 ]
Test mIoU all: 0.5743
Test target mIoU: 0.4721


In [16]:
@torch.no_grad()
def get_confusion_matrix(
    model,
    loader,
    device,
    num_classes=4,
    ignore_index=255,
):
    model.eval()

    confusion = torch.zeros(
        num_classes,
        num_classes,
        dtype=torch.int64,
        device=device,
    )

    for images, masks in loader:
        images = images.to(
            device,
            memory_format=torch.channels_last,
            non_blocking=True,
        )

        masks = masks.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=amp_dtype,
        ):
            logits = model(images)

        preds = logits.argmax(dim=1)

        valid = masks != ignore_index

        y_true = masks[valid].long()
        y_pred = preds[valid].long()

        indices = y_true * num_classes + y_pred

        confusion += torch.bincount(
            indices,
            minlength=num_classes * num_classes,
        ).reshape(num_classes, num_classes)

    return confusion.cpu().numpy()

In [23]:
cm = get_confusion_matrix(
    model=trained_model,
    loader=test_loader,
    device=device,
)

print("Confusion matrix:")
print(cm)

Confusion matrix:
[[3079631  292060   93335   18791]
 [  11108   91973    7066     133]
 [    241   11517  159049    4170]
 [      0       0    8101   53797]]
